<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day2/notebooks/1_bow_approaches_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 2 — Bag-of-Words Approaches  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible
answer. Because sentiment and topics are subjective, your interpretations may reasonably
differ from the comments here.


## 0. Setup

In [ ]:
!pip install datasets wordcloud -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import LatentDirichletAllocation

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")

print("Setup complete!")

## 1. Loading the TweetEval sentiment data

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Train size:", len(train_df))
print("Test size: ", len(test_df))
train_df.head()

In [ ]:
label_names = {0: "negative", 1: "neutral", 2: "positive"}
train_df["label_name"] = train_df["label"].map(label_names)
test_df["label_name"] = test_df["label"].map(label_names)

print("Label distribution (train):")
print(train_df["label_name"].value_counts())
print()
for lab in [0, 1, 2]:
    example = train_df[train_df["label"] == lab]["text"].iloc[0]
    print(f"[{label_names[lab]}] {example}")

> **✏️ Exercise 1**
>
> Print **three** example tweets for the *negative* class (label 0). Do the human labels
> look reasonable?


In [ ]:
# ✅ Solution
neg_examples = train_df[train_df["label"] == 0]["text"].head(3)
for i, tweet in enumerate(neg_examples, 1):
    print(f"{i}. {tweet}")
    print()

# Comment: some will look clearly negative; others may seem neutral or ambiguous to you.
# That disagreement is exactly why sentiment labeling is hard — and why no method
# (dictionary or ML) can be perfectly "right" against labels that humans themselves
# dispute.

### A note on size

In [ ]:
SAMPLE_SIZE = 3000
train_sample = train_df.sample(n=min(SAMPLE_SIZE, len(train_df)), random_state=42).reset_index(drop=True)
test_sample = test_df.sample(n=min(1000, len(test_df)), random_state=42).reset_index(drop=True)
print("Working sample sizes:")
print("  Train:", len(train_sample))
print("  Test: ", len(test_sample))

## 2. Light preprocessing

In [ ]:
import re

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    return " ".join(tokens)

print("RAW:    ", train_sample["text"].iloc[0])
print("CLEANED:", clean_tweet(train_sample["text"].iloc[0]))

In [ ]:
train_sample["clean"] = train_sample["text"].apply(clean_tweet)
test_sample["clean"] = test_sample["text"].apply(clean_tweet)
print("Done cleaning.")

## 3. Approach 1 — Dictionary methods

In [ ]:
positive_words = {
    "good", "great", "love", "happy", "excellent", "wonderful", "best",
    "amazing", "awesome", "nice", "beautiful", "fantastic", "perfect",
    "thanks", "thank", "glad", "excited", "enjoy", "win", "congratulations",
}
negative_words = {
    "bad", "hate", "terrible", "awful", "worst", "horrible", "sad", "angry",
    "disappointed", "disappointing", "poor", "ugly", "wrong", "fail", "sucks",
    "boring", "annoying", "useless", "waste", "sorry",
}

def dictionary_sentiment(text):
    tokens = text.split()
    score = sum(1 for t in tokens if t in positive_words)
    score -= sum(1 for t in tokens if t in negative_words)
    if score > 0:
        return 2
    elif score < 0:
        return 0
    else:
        return 1

test_sample["dict_pred"] = test_sample["clean"].apply(dictionary_sentiment)
test_sample[["text", "label_name", "dict_pred"]].head(8)

In [ ]:
acc = accuracy_score(test_sample["label"], test_sample["dict_pred"])
print(f"Dictionary accuracy: {acc:.3f}")
print()
print(classification_report(
    test_sample["label"], test_sample["dict_pred"],
    target_names=["negative", "neutral", "positive"],
    zero_division=0,
))

> **✏️ Exercise 2**
>
> Add a few of your own words, re-run the scoring, and see whether accuracy improves. What
> are the limits of just adding more words?


In [ ]:
# ✅ Solution
positive_words |= {"brilliant", "superb", "grateful", "delighted", "recommend", "worth"}
negative_words |= {"disgusting", "pathetic", "furious", "regret", "broken", "scam"}

test_sample["dict_pred"] = test_sample["clean"].apply(dictionary_sentiment)
acc2 = accuracy_score(test_sample["label"], test_sample["dict_pred"])
print(f"Dictionary accuracy after adding words: {acc2:.3f}")

# Comment on the limits:
# Adding words gives diminishing returns and can even hurt. The real problems are
# structural, not vocabulary size:
#   - negation ("not good") is invisible to a word-counting approach
#   - sarcasm/irony flips meaning entirely
#   - the same word carries different sentiment in different contexts
# No amount of list-expansion fixes these — that needs a model that uses context.

**Reflection.** The dictionary is transparent but context-blind.

## 4. Approach 2 — Supervised machine learning

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_sample["clean"])
X_test = vectorizer.transform(test_sample["clean"])

y_train = train_sample["label"]
y_test = test_sample["label"]

print("Training matrix shape:", X_train.shape)

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(
    y_test, y_pred,
    target_names=["negative", "neutral", "positive"],
    zero_division=0,
))

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
for class_idx, class_name in enumerate(["negative", "neutral", "positive"]):
    top = np.argsort(clf.coef_[class_idx])[-10:][::-1]
    print(f"Top words for '{class_name}':")
    print("  ", ", ".join(feature_names[top]))
    print()

> **✏️ Exercise 3**
>
> Swap in **Multinomial Naive Bayes** and compare accuracy. Which does better?


In [ ]:
# ✅ Solution
nb_clf = MultinomialNB()
nb_clf.fit(X_train, y_train)
nb_pred = nb_clf.predict(X_test)

print(f"Logistic Regression accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Naive Bayes accuracy:         {accuracy_score(y_test, nb_pred):.3f}")

# Comment: on this dataset logistic regression usually edges out Naive Bayes,
# but they are often close. Naive Bayes is faster and a strong baseline; logistic
# regression tends to handle correlated features a bit better. "Which is better"
# is always empirical — try both.

### Visualizing errors: the confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Greens")
labels = ["negative", "neutral", "positive"]
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (supervised classifier)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im)
plt.tight_layout()
plt.show()

> **✏️ Exercise 4**
>
> Which class is hardest for the model?


In [ ]:
# ✅ Solution
# Per-class recall tells us how often each true class is correctly caught.
from sklearn.metrics import recall_score
recalls = recall_score(y_test, y_pred, average=None)
for name, r in zip(["negative", "neutral", "positive"], recalls):
    print(f"{name:>9}: recall = {r:.3f}")

# Comment: 'neutral' is typically the hardest. Neutral tweets lack the strong
# sentiment-bearing words the model leans on, and they sit "between" the other two
# classes, so they are easily pulled toward negative or positive. You can usually
# see this in the confusion matrix as neutral being spread across the other columns.

## 5. Approach 3 — Topic modeling (unsupervised)

In [ ]:
count_vec = CountVectorizer(
    max_features=1000,
    stop_words="english",
    min_df=5,
)
X_counts = count_vec.fit_transform(train_sample["clean"])
print("Matrix for topic modeling:", X_counts.shape)

In [ ]:
N_TOPICS = 5
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=10)
lda.fit(X_counts)
print(f"Fitted an LDA model with {N_TOPICS} topics.")

In [ ]:
def show_topics(model, feature_names, n_words=10):
    for idx, topic in enumerate(model.components_):
        top = topic.argsort()[-n_words:][::-1]
        words = [feature_names[i] for i in top]
        print(f"Topic {idx}: {', '.join(words)}")

feature_names = count_vec.get_feature_names_out()
show_topics(lda, feature_names)

> **✏️ Exercise 5**
>
> Give each topic a **human label**.


In [ ]:
# ✅ Solution (example — yours will differ depending on the sample)
# Reading the top words, you might label them something like:
#   Topic 0 -> everyday chat / logistics
#   Topic 1 -> positive reactions
#   Topic 2 -> negative complaints
#   Topic 3 -> news / events
#   Topic 4 -> mixed / hard to interpret (junk topic)
#
# It is completely normal for at least one topic to be incoherent. Labeling is an
# interpretive act — two researchers might reasonably disagree. Document your reasoning.
print("See comments — interpretation is qualitative work.")

> **✏️ Exercise 6**
>
> Re-fit LDA with `N_TOPICS = 3` and `N_TOPICS = 8`. How does interpretability change?


In [ ]:
# ✅ Solution
for k in [3, 8]:
    print(f"\n===== {k} topics =====")
    lda_k = LatentDirichletAllocation(n_components=k, random_state=42, max_iter=10)
    lda_k.fit(X_counts)
    show_topics(lda_k, feature_names, n_words=8)

# Comment: fewer topics (3) gives broad, blended themes that are easy to label but
# lose nuance. More topics (8) gives finer distinctions but more overlap and more
# junk topics that are hard to name. There is no single correct number — you choose
# based on interpretability and your research question, sometimes guided by metrics
# like coherence.

## 6. Putting it together

### Optional challenge

Find a tweet where the **dictionary got it wrong because of negation or sarcasm**.


In [ ]:
# ✅ Solution
# Look for tweets the dictionary misclassified, then scan for negation words.
test_sample["dict_pred"] = test_sample["clean"].apply(dictionary_sentiment)
wrong = test_sample[test_sample["dict_pred"] != test_sample["label"]]

negation_words = {"not", "no", "never", "n't", "without"}
for _, row in wrong.iterrows():
    tokens = set(row["clean"].split())
    if tokens & negation_words:
        print("TWEET:", row["text"])
        print("  true:", label_names[row["label"]],
              "| dict predicted:", label_names[row["dict_pred"]])
        print()
        break

# Comment: the dictionary counts sentiment words but ignores the "not"/"never" that
# flips them. "not good" contributes +1 for "good" and nothing for "not", so it scores
# positive. Embeddings (and especially contextual models like BERT) can represent
# "not good" differently from "good" — which is exactly why we move to them next.